[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/polars-certified/notebooks/day-08-performance-optimization-and-profiling.ipynb#scrollTo=b1c2d3e4)

---
# Day 8 · Performance Optimization and Profiling
**certified-journeys / polars-certified** &nbsp;|&nbsp; Advanced

> **Goal for today:** Profile Polars queries, use streaming for out-of-core processing, understand memory layout, and benchmark correctly.

---
## The Polars performance model

Polars is fast for three compounding reasons:

| Layer | What it does | Your lever |
|---|---|---|
| **Rust execution engine** | SIMD, zero-copy, cache-friendly loops | Use expressions (not UDFs) |
| **Lazy optimizer** | Predicate pushdown, projection pruning, CSE | Use `LazyFrame`, check `.explain()` |
| **Streaming engine** | Out-of-core, bounded memory | `.collect(streaming=True)` |

Today you will measure each layer and learn how to diagnose regressions.

In [ ]:
%pip install -q polars pyarrow pandas

---
## Step 1 · .explain() — reading the query plan

`.explain()` shows you what Polars **will** execute before it executes it.
The optimized plan shows what the lazy optimizer has rearranged — always compare both.

In [ ]:
import polars as pl
import numpy as np
import time
import tempfile, os, pathlib

rng = np.random.default_rng(0)
N = 1_000_000  # 1 million rows for benchmarking

# Generate a 1M-row synthetic dataset in memory
df = pl.DataFrame({
    "id":       np.arange(N),
    "category": rng.choice(["A", "B", "C", "D"], N).tolist(),
    "value":    rng.uniform(0.0, 1000.0, N),
    "flag":     rng.integers(0, 2, N).astype(bool).tolist(),
})

# Build a lazy pipeline with filter + select + aggregation
lf = (
    df.lazy()
    .filter(pl.col("flag") == True)           # predicate
    .select(["category", "value"])             # projection (drop 'id' and 'flag')
    .group_by("category")
    .agg(pl.col("value").mean().alias("avg_value"))
)

print("=== UNOPTIMIZED PLAN ===")
print(lf.explain(optimized=False))

print("\n=== OPTIMIZED PLAN ===")
print(lf.explain(optimized=True))

**What just happened?**
- The **unoptimized plan** shows operations in the order you wrote them: filter, then select, then group_by.
- The **optimized plan** shows Polars has **pushed the projection** (column selection) earlier — it reads fewer columns from the start.
- Look for `FILTER` near the data source in the optimized plan: that's **predicate pushdown** reducing rows early.
- **Rule of thumb:** if your optimized plan looks identical to the unoptimized plan, check whether you're using Polars expressions (good) or Python UDFs (bad — they block optimization).

---
## Step 2 · Streaming mode for out-of-core processing

`.collect(streaming=True)` processes data in bounded-size chunks rather than loading
everything into RAM at once. Essential when your dataset exceeds available memory.

In [ ]:
TMPDIR = tempfile.mkdtemp()
parquet_path = os.path.join(TMPDIR, "large.parquet")
df.write_parquet(parquet_path)

# Standard lazy collect (all data in RAM)
t0 = time.perf_counter()
result_standard = (
    pl.scan_parquet(parquet_path)
    .filter(pl.col("flag") == True)
    .group_by("category")
    .agg(pl.col("value").mean())
    .collect()
)
standard_ms = (time.perf_counter() - t0) * 1000
print(f"Standard collect:   {standard_ms:.1f} ms  |  result shape: {result_standard.shape}")

# Streaming collect (bounded RAM, works on datasets larger than memory)
t0 = time.perf_counter()
result_streaming = (
    pl.scan_parquet(parquet_path)
    .filter(pl.col("flag") == True)
    .group_by("category")
    .agg(pl.col("value").mean())
    .collect(streaming=True)
)
streaming_ms = (time.perf_counter() - t0) * 1000
print(f"Streaming collect:  {streaming_ms:.1f} ms  |  result shape: {result_streaming.shape}")

print("""
Note: streaming may be slower on small datasets (chunk overhead).
Its advantage is constant memory use regardless of dataset size.
Not all operations support streaming (e.g., sort without limit does not).
""")

**What just happened?**
- Streaming mode processes data in **fixed-size chunks** — peak RAM stays constant no matter how large the file.
- On small datasets, streaming has overhead and may be slower than standard collect — this is normal.
- **Operations that support streaming**: filter, select, group_by+agg, join (some modes), scan_parquet/scan_csv.
- **Operations that block streaming**: `sort()` without a `slice`, `unique()` on large columns, some window functions. Polars falls back to standard mode automatically when streaming is unsupported.

---
## Step 3 · Rechunking and memory layout

Polars stores data in **chunks** (Arrow arrays). Many concatenation operations create
fragmented multi-chunk layouts that hurt performance. `rechunk()` consolidates them.

In [ ]:
# Simulate a fragmented DataFrame by concatenating many small pieces
pieces = [pl.DataFrame({"x": [i] * 1000, "y": [float(i)] * 1000}) for i in range(20)]
fragmented = pl.concat(pieces, rechunk=False)  # rechunk=False keeps fragmentation

print(f"Fragmented chunks:  {fragmented.n_chunks()}")
print(f"Shape:              {fragmented.shape}")

# rechunk() collapses all chunks into one contiguous block
consolidated = fragmented.rechunk()
print(f"After rechunk:      {consolidated.n_chunks()} chunk")

# Memory footprint: Categorical vs String
many_strings = pl.DataFrame({
    "region_str": ["US", "EU", "APAC", "LATAM"] * 250_000,  # 1M rows
})
many_cat = many_strings.with_columns(
    pl.col("region_str").cast(pl.Categorical).alias("region_cat")
)

str_size = many_strings.estimated_size() / 1_048_576
cat_size  = many_cat.select("region_cat").estimated_size() / 1_048_576
print(f"\nString column (1M rows):     {str_size:.2f} MB")
print(f"Categorical column (1M rows): {cat_size:.2f} MB")
print(f"Memory saving: {str_size/cat_size:.1f}x")

**What just happened?**
- `n_chunks()` reveals memory fragmentation — 20 concatenated pieces = 20 chunks, each requiring separate iteration.
- `rechunk()` consolidates into **1 contiguous chunk** — cache-friendly iteration and faster subsequent operations.
- **Categorical columns** store each unique string once and use integer indices — dramatically smaller for low-cardinality columns like `region`, `status`, `country`.
- **Rule:** call `rechunk()` after many `concat()` operations, or after building a DataFrame incrementally in a loop.

---
## Step 4 · Verbose optimizer messages

Setting `POLARS_VERBOSE=1` before running Polars prints internal optimizer decisions to stderr.
This is useful for debugging why a query is slower than expected.

```bash
# In a terminal or Colab cell:
POLARS_VERBOSE=1 python my_script.py

# In a notebook, set before import:
import os
os.environ["POLARS_VERBOSE"] = "1"
import polars as pl
```

You'll see messages like:
- `projection pushdown: [...]` — which columns were pruned
- `predicate pushdown: [...]` — which filters moved to the scan
- `cse: replaced N subexpressions` — common sub-expression elimination

**When to use it:** when `.explain()` shows the optimization you expect but the query is still slow — verbose mode reveals whether the optimizer is actually applying the optimization at runtime.

---
## Step 5 · Benchmarking: Polars eager vs lazy vs Pandas

Good benchmarks avoid cold-start bias. We warm up the function once,
then time it in a loop and take the minimum (not mean) — the minimum removes OS scheduling noise.

In [ ]:
import pandas as pd

df_pd = df.to_pandas()  # Pandas version of the same 1M-row dataset
REPS = 5                # number of timed repetitions

def bench(fn, reps=REPS):
    """Warm up once, then take the minimum of 'reps' runs (ms)."""
    fn()  # warm-up to avoid first-run JIT / I/O cache effects
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    return min(times)

# 1. Polars eager
def polars_eager():
    return (
        df
        .filter(pl.col("flag") == True)
        .group_by("category")
        .agg(pl.col("value").mean())
    )

# 2. Polars lazy
def polars_lazy():
    return (
        df.lazy()
        .filter(pl.col("flag") == True)
        .group_by("category")
        .agg(pl.col("value").mean())
        .collect()
    )

# 3. Pandas equivalent
def pandas_agg():
    return (
        df_pd[df_pd["flag"] == True]
        .groupby("category")["value"]
        .mean()
        .reset_index()
    )

t_eager  = bench(polars_eager)
t_lazy   = bench(polars_lazy)
t_pandas = bench(pandas_agg)

print(f"Polars eager:  {t_eager:.1f} ms")
print(f"Polars lazy:   {t_lazy:.1f} ms")
print(f"Pandas:        {t_pandas:.1f} ms")
print(f"\nPolars lazy is {t_pandas/t_lazy:.1f}x faster than Pandas")

**What just happened?**
- We warm up once before timing — this avoids measuring Python import overhead or OS file-cache misses.
- `min()` over multiple runs removes scheduling jitter; `mean()` would be polluted by occasional slow runs.
- **Polars lazy is typically faster than eager** because the optimizer can reorder and prune operations.
- **Polars vs Pandas**: the gap depends on the operation — aggregations and filters show the largest speedup because Polars uses SIMD and multi-threading; Pandas is single-threaded.

---
## Step 6 · pl.Config for display tuning

`pl.Config` lets you adjust how DataFrames are displayed in notebooks and terminals.
Useful for large DataFrames and float formatting.

In [ ]:
# Show only 5 rows in repr (default is 10)
pl.Config.set_tbl_rows(5)

# Display floats with 2 decimal places
pl.Config.set_fmt_float("full")   # or "mixed"

sample = pl.DataFrame({"x": [1.23456789, 2.987654, 0.00001, 100.0, 999.999]})
print("Config-controlled display:")
print(sample)

# Show estimated memory size for a realistic DataFrame
size_mb = df.estimated_size() / 1_048_576
print(f"\n1M-row DataFrame estimated size: {size_mb:.2f} MB")

# Compare Categorical vs String memory for the 'category' column
str_mb = df.select("category").estimated_size() / 1_048_576
cat_mb = df.with_columns(pl.col("category").cast(pl.Categorical)).select("category").estimated_size() / 1_048_576
print(f"category as String:      {str_mb:.3f} MB")
print(f"category as Categorical: {cat_mb:.3f} MB  ({str_mb/cat_mb:.1f}x smaller)")

# Reset to defaults
pl.Config.set_tbl_rows(10)

**What just happened?**
- `pl.Config.set_tbl_rows(N)` controls the repr row limit — essential in notebooks to avoid flooding output.
- `estimated_size()` gives a fast approximation of in-memory size without calling `sys.getsizeof` on each element.
- **Categorical is dramatically smaller** for columns with few unique values — 4 categories across 1M rows stores just 4 strings + 1M integers instead of 1M string objects.
- Use `estimated_size()` before and after dtype changes to verify memory savings are real.

---
## Challenge

Take an unoptimised **eager** pipeline with 4 chained operations and convert it to a **lazy** pipeline.
Measure the speedup with the benchmarking pattern from Step 5.

In [ ]:
# Unoptimised eager pipeline (given)
def unoptimised():
    # Operation 1: load all columns and all rows eagerly
    step1 = df.filter(pl.col("value") > 500.0)        # filter
    # Operation 2: select only the columns we need
    step2 = step1.select(["category", "value"])        # projection
    # Operation 3: aggregate
    step3 = step2.group_by("category").agg(pl.col("value").sum())
    # Operation 4: sort results
    step4 = step3.sort("value", descending=True)
    return step4

# TODO: convert the same logic to a lazy pipeline
def optimised_lazy():
    return ...  # df.lazy().filter(...).select(...).group_by(...).agg(...).sort(...).collect()

# TODO: benchmark both and print the speedup
# t_slow = bench(unoptimised)
# t_fast = bench(optimised_lazy)
# print(f"Eager:  {t_slow:.1f} ms")
# print(f"Lazy:   {t_fast:.1f} ms  ({t_slow/t_fast:.1f}x speedup)")

---
## Day 8 recap

| Concept | Key method | When to use |
|---|---|---|
| Query plan inspection | `lf.explain(optimized=True)` | Always before optimising |
| Streaming execution | `.collect(streaming=True)` | Datasets larger than RAM |
| Memory fragmentation | `df.n_chunks()` / `df.rechunk()` | After many concat operations |
| Memory estimation | `df.estimated_size()` | Compare dtypes, verify savings |
| Low-cardinality strings | `cast(pl.Categorical)` | region, status, country columns |
| Benchmarking | `time.perf_counter()` + warm-up + `min()` | Comparing approaches |
| Display config | `pl.Config.set_tbl_rows()` | Notebook presentation |

> **Tip:** Streaming mode (`collect(streaming=True)`) lets Polars process datasets larger than RAM. Essential for production. Always check the optimized plan with `.explain(optimized=True)` first — if the plan looks right but performance is still poor, check `n_chunks()` for memory fragmentation.

---
**What's next → Day 9:** Custom functions, plugins and Python UDFs — when (and when not) to use them, and how to benchmark the difference.

Mark Day 8 complete in your [tracker](../index.html).